# Chapter 4 &mdash; Totalization and the Black-Hole State

**Concept 5 of the Chapter 4 decomposition:** *Totality of $\delta$, Totalization, and the Black-Hole State*

Every state must answer every symbol. Missing moves go to a sink you can enter but never leave.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-Totalization-Black-Hole/Concept-Totalization-Black-Hole.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


$\delta$ is **total**, but it is convenient to specify a DFA *partially* and then
**totalize** it &mdash; routing every unspecified move to a **black-hole** state.

The synonyms are worth knowing because the literature is inconsistent: *Roach-Motel*
("you can check in, you can't check out"), *sink*, *empty* state.

A DFA must classify **every** string in $\Sigma^*$, so rejection is a **destination**,
not an absence.

## 2. Definitions

### A partially specified machine, then totalized

In [ ]:
partial = md2mc('''DFA
IF : 0 -> A
A  : 0 -> IF
''')
print("Sigma before :", sorted(partial["Sigma"]))
wider = addtosigma_dfa(partial, {'1'})
total = totalize_dfa(wider)
print("Sigma after  :", sorted(total["Sigma"]))
print("states after :", sorted(total["Q"]), " <- note the new sink")

### Finding the black hole

A sink is a non-final state whose every move returns to itself.

In [ ]:
def black_holes(D):
    return [q for q in D["Q"]
            if q not in D["F"]
            and all(step_dfa(D, q, c) == q for c in D["Sigma"])]

## 3. Tests

Order matters: widen the alphabet **before** totalizing.

In [ ]:
print("black holes :", black_holes(total))
assert black_holes(total), "totalizing should have added a sink"

Once in the sink, you never leave &mdash; so the string is rejected whatever follows.

In [ ]:
bh = black_holes(total)[0]
print("from", bh, "on '0' ->", step_dfa(total, bh, '0'))
print("from", bh, "on '1' ->", step_dfa(total, bh, '1'))
assert all(step_dfa(total, bh, c) == bh for c in total["Sigma"])

Every string now gets a verdict &mdash; that is what totality buys.

In [ ]:
for s in ['', '00', '1', '001', '0100']:
    print("%-7r accepted? %s" % (s, accepts_dfa(total, s)))
print("\nNo string is left undecided. Rejection is a destination, not an absence.")

## 4. Animation

`dotObj_dfa` hides black holes by default; the animation shows the totalized machine, sink and all.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(total, FuseEdges=True)

## 5. Exercises


1. Totalize **before** widening the alphabet instead. What goes wrong?
2. `dotObj_dfa_w_bh` draws sinks; `dotObj_dfa` hides them. Draw both and compare.
3. Why must complementation (Chapter 6) totalize first?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 245 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
from jove.Nav import nav, load_here
nav(here='Chapter4/Concept-Totalization-Black-Hole')